# Uber SQL Database - Unit 4 Practice (Solutions)

This notebook contains 10 WJEC-style Unit 4 SQL exercises based on the `rockyt07/uber-sql-database` dataset.
Each exercise includes a prompt and a validated SQL solution using `%%sql`.

Constraint for this workbook: any query that returns rows should return no more than 6 rows.
Use filtering and `LIMIT 6` where appropriate so outputs are manageable and easy to verify.

In [2]:
from pathlib import Path
import sys

notebooks_root = Path.cwd()
while not (notebooks_root / "helpers").exists() and notebooks_root != notebooks_root.parent:
    notebooks_root = notebooks_root.parent

sys.path.insert(0, str(notebooks_root))

from helpers import setup_uber_sql_notebook

setup_uber_sql_notebook()

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.
✓ Setup complete! Database ready.


PosixPath('/workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db')

## ⚠️ IMPORTANT: Run the setup code cell above FIRST

Before viewing or running the exercises below, execute the setup code cell above by clicking the play button or pressing Shift+Enter. This single cell will:
1. Download the Uber rideshare dataset from Kaggle
2. Load the reusable SQL setup helper and connect to the database
3. Verify that the database is ready for exercises

Once you see the **"✓ Setup complete! Database ready."** message, all exercises and solutions are ready to view and run.

In [6]:
%%sql
SELECT trip_id, status, total_fare
FROM trips
WHERE status = 'completed'
  AND total_fare > 30
ORDER BY total_fare DESC
LIMIT 6;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


trip_id,status,total_fare
19709,completed,224.27
5380,completed,222.4
8306,completed,220.58
13361,completed,216.23
4156,completed,213.59
12783,completed,213.36


## Exercise 2 - Active drivers profile (DQL: SELECT, WHERE)

Display up to 6 active drivers (`is_active = 1`) with rating at least 4.5.
Show `driver_id`, `vehicle_make`, `vehicle_model`, and `rating` in descending rating order.

In [30]:
%%sql
SELECT driver_id, vehicle_make, vehicle_model, rating
FROM drivers
WHERE is_active = 1
  AND rating >= 4.5
ORDER BY rating DESC, driver_id ASC
LIMIT 6;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


driver_id,vehicle_make,vehicle_model,rating
35,Chevrolet,Silverado,5.0
178,Mazda,MX-5,5.0
207,Honda,CR-V,4.99
260,Ford,Escape,4.99
298,Tesla,Model X,4.99
326,Nissan,Rogue,4.99


## Exercise 3 - Successful electronic transactions (DQL: IN, WHERE, ORDER BY)

Find up to 6 successful payments made by card or wallet.
Show `payment_id`, `trip_id`, `amount`, `method`, and `status`, ordered by highest amount first.

In [41]:
%%sql
SELECT payment_id, trip_id, amount, method, status
FROM payments
WHERE status = 'success'
  AND method IN ('card', 'wallet')
ORDER BY amount DESC
LIMIT 6;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


payment_id,trip_id,amount,method,status
16584,19709,224.27,card,success
11221,13361,216.23,card,success
3485,4156,213.59,card,success
10734,12783,213.36,wallet,success
9606,11464,204.16,wallet,success
15900,18895,204.05,wallet,success


## Exercise 4 - Average successful payment by method (DQL: GROUP BY)

For successful (`success`) payments, calculate the number of payments and average amount per payment method.
Return `method`, `payment_count`, and `avg_amount` (up to 6 rows).

In [32]:
%%sql
SELECT method,
       COUNT(*) AS payment_count,
       ROUND(AVG(amount), 2) AS avg_amount
FROM payments
WHERE status = 'success'
GROUP BY method
ORDER BY avg_amount DESC
LIMIT 6;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


method,payment_count,avg_amount
card,5549,36.53
wallet,5450,35.55
cash,5360,35.5


## Exercise 5 - Rider names on expensive completed trips (Relational: JOIN)

List the 6 highest-fare completed trips with rider names.
Show `trip_id`, `rider_name`, `driver_id`, and `total_fare`.

In [33]:
%%sql
SELECT t.trip_id,
       u.name AS rider_name,
       t.driver_id,
       t.total_fare
FROM trips AS t
JOIN riders AS r ON t.rider_id = r.rider_id
JOIN users AS u ON r.user_id = u.user_id
WHERE t.status = 'completed'
ORDER BY t.total_fare DESC
LIMIT 6;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


trip_id,rider_name,driver_id,total_fare
19709,Debra Howard,24,224.27
5380,Eric Hughes,320,222.4
8306,Catherine Rogers,35,220.58
13361,David Parker,55,216.23
4156,Timothy Richardson,30,213.59
12783,Betty Gray,66,213.36


## Exercise 6 - Above-average driver ratings (Relational: subquery)

Show up to 6 drivers whose rating is above the average driver rating.
Return `driver_id` and `rating` sorted by highest rating.

In [34]:
%%sql
SELECT driver_id, rating
FROM drivers
WHERE rating > (
    SELECT AVG(rating)
    FROM drivers
    WHERE rating IS NOT NULL
)
ORDER BY rating DESC, driver_id ASC
LIMIT 6;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


driver_id,rating
35,5.0
178,5.0
207,4.99
260,4.99
298,4.99
326,4.99


## Exercise 7 - Most-used pickup zones (Relational: JOIN + GROUP BY)

Find the 6 pickup zones used most often.
Return `zone_name` and `trip_count` ordered by highest trip count first.

In [35]:
%%sql
SELECT l.zone_name,
       COUNT(*) AS trip_count
FROM trips AS t
JOIN locations AS l ON t.pickup_location_id = l.location_id
GROUP BY l.location_id, l.zone_name
ORDER BY trip_count DESC
LIMIT 6;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


zone_name,trip_count
Flushing,607
Greenwich Village,593
Upper West Side,593
Times Square,582
JFK Airport,578
Downtown Houston,568


## Exercise 8 - Create a promotions table (DDL: CREATE TABLE)

Create a new table for promo codes with:
- `promo_id` as `PRIMARY KEY`
- `code` as `NOT NULL`
- `discount_pct` as `NOT NULL`
- `active` as `NOT NULL`

In [36]:
%%sql
CREATE TABLE IF NOT EXISTS practice_promotions (
    promo_id INTEGER PRIMARY KEY,
    code TEXT NOT NULL,
    discount_pct REAL NOT NULL,
    active INTEGER NOT NULL
);

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


[]

## Exercise 9 - Insert promo data (DML: INSERT)

Insert three records into `practice_promotions` using these values:
1. `(1, 'SPRING10', 10.0, 1)`
2. `(2, 'RIDER5', 5.0, 1)`
3. `(3, 'EXPIRED20', 20.0, 0)`

In [37]:
%%sql
INSERT OR REPLACE INTO practice_promotions (promo_id, code, discount_pct, active) VALUES
    (1, 'SPRING10', 10.0, 1),
    (2, 'RIDER5', 5.0, 1),
    (3, 'EXPIRED20', 20.0, 0);

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
3 rows affected.


[]

## Exercise 10 - Update promo status (DML: UPDATE)

Set the promo code `EXPIRED20` to inactive (`active = 0`) and then display all rows from `practice_promotions` to verify the change.

In [38]:
%%sql
UPDATE practice_promotions
SET active = 0
WHERE code = 'EXPIRED20';

SELECT *
FROM practice_promotions
ORDER BY promo_id;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
1 rows affected.
Done.


promo_id,code,discount_pct,active
1,SPRING10,10.0,1
2,RIDER5,5.0,1
3,EXPIRED20,20.0,0
